# KNN Experiment

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation import (
    cross_validate_configs,
    generate_submission,
    save_and_verify_model,
    split_train_validation,
    train_and_evaluate,
    verify_smote_order,
)
from src.models import DEFAULT_KNN_CONFIGS, KNN, get_knn_preprocessing_config
from src.preprocess.preprocess import run_preprocessing

RANDOM_STATE = 824


In [ ]:
data_dir = REPO_ROOT / "data"
train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

prep = run_preprocessing(
    train_df,
    test_df,
    config=get_knn_preprocessing_config(random_state=RANDOM_STATE, verbose=True),
)


In [ ]:
verify_smote_order(prep.X_train_encoded, prep.X_train_balanced, verbose=True)
split_data = split_train_validation(
    prep.X_train,
    prep.y_train,
    is_orig=prep.is_orig,
    test_size=0.15,
    random_state=RANDOM_STATE,
    verbose=True,
)

best_knn_config, knn_cv_df = cross_validate_configs(
    KNN,
    DEFAULT_KNN_CONFIGS,
    split_data.X_tr,
    split_data.y_tr,
    k_folds=5,
    random_state=RANDOM_STATE,
    model_name="KNN",
    verbose=True,
)
display(knn_cv_df)

knn_result = train_and_evaluate(
    KNN(**best_knn_config),
    split_data,
    model_name="KNN",
    config=best_knn_config,
    X_test=prep.X_test,
    plot_cm=True,
)


In [ ]:
output_dir = REPO_ROOT / "output"
loaded_knn = save_and_verify_model(knn_result["model"], output_dir / "knn.pkl", verbose=True)
submission = generate_submission(
    loaded_knn,
    knn_result["X_test"],
    test_df["id"],
    sample_submission,
    output_dir / "submission_knn.csv",
    random_state=RANDOM_STATE,
    verbose=True,
)
display(submission.head())
